In [2]:
from pathlib import Path
home = Path.home()
shared = Path('/home/shared/owner')
root = shared/'CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM'

In [ ]:
from numpy import pi, diff, cumsum
from ior2ecl import IORSim_output
from IORlib.ECL import UNSMRY_file

stb2m3 = 1.0 / 6.289811                # 0.158987294928 m3
stb_day2cm3_hour = stb2m3 * 1e6 / 24   # 1 m3 = 1e6 cm3

dia = 3.8  # cm
L = 22.4   # cm
porosity = 0.4
vol = L * pi * (0.5*dia)**2  # cm^3
porvol = porosity * vol      # cm^3

smry = UNSMRY_file(root)
ix = smry.welldata(wells=('Prod_Com',), keys=('WWPR',), named=True, as_array=True)
ior_no = IORSim_output(root).welldata('Prod_Com', path='ior_nocorr')
ior = IORSim_output(root).welldata('Prod_Com', path='ior_corr')

#print(len(ix.days), len(ior.days))
# Convert to porevolume
rate = ix.WWPR.Prod_Com * stb_day2cm3_hour
# Skip first ix data to be in sync with ior data, i.e. ix.days[1] == ior.days[0]
rate = rate[1:]
# For some reason ior step 129 is out of sync with the ix steps
#list(zip(range(1000), ix.days[1:], ior.days))
dt = diff(ior.days)*24
#PV = cumsum(rate[1:]*dt)/porvol
PV = cumsum(rate*dt)/porvol

In [ ]:
import matplotlib.pyplot as pl
from IORlib.utils import molL2ppm
%matplotlib widget
pl.close('all')
species = ['H', 'Na', 'K', 'Ca', 'Mg', 'Cl', 'SO4', 'HCO3']
#for sp, c in zip(ior_noior.conc.items():
for specie in species:
    SPECIE = specie.upper()
    pl.figure()
    pl.title(specie)
    pl.xlabel('Pore volumes injected')
    pl.ylabel('PPM')
    pl.semilogy(PV, molL2ppm(specie) * ior.conc[SPECIE][1:], '-o', mfc='None', ms=4, label='Corr')
    pl.semilogy(PV, molL2ppm(specie) * ior_no.conc[SPECIE][1:], '-o', mfc='None', ms=4, label='No corr')


In [ ]:
import matplotlib.pyplot as pl
from ior2ecl import IORSim_output
data = IORSim_output(root).welldata('Prod_Com')
#pl.plot(data.days, data.conc['CL'], '-o', mfc='None')
pl.close('all')
for specie, conc in data.conc.items():
    pl.figure()
    pl.title(specie)
    pl.plot(data.days, conc, '-o', mfc='None')

In [ ]:
from IORlib.ECL import UNSMRY_file
smry = UNSMRY_file(root)
pl.close('all')
smry.plot(keys=('WOPR', 'WWPR','WWIR'), args={'linestyle':'-', 'ms':5, 'mfc':'None'})

### Rates from CON*TUB
The following code calculate well connection rates from CON*TUB values based on the connection neighbour list in CONNXT. Data are taken from the RFT-file

This code is implemented in IORSim

In [3]:
from itertools import batched, islice
from more_itertools import flatten
from IORlib.ECL import RFT_file, File, IX_input, UNRST_file, UNSMRY_file
from IORlib.utils import missing_elements
from numpy import array, max as npmax, diff as npdiff, array, argsort, ones, bool_, stack, zeros, atleast_1d, squeeze, sum as npsum
from pprint import PrettyPrinter

pp = PrettyPrinter(sort_dicts=False)
barrel2m3 = 1.0 / 6.289811 # 0.158987294928 = inch**3*42*231
mscf2m3 = 28.316846592 # = 1e3*0.3048**3 # mscf = thousand standard cubic feet
mscf2barrel = mscf2m3/barrel2m3

def rate_from_tubs(tub, wellrate, connxt):
    connxt = atleast_1d(connxt)
    tub = atleast_1d(tub)
    injector = wellrate < 0
    idx = list(argsort(connxt))
    miss = missing_elements(connxt) or [len(connxt)]
    is_neigh = ones(len(idx) + len(miss), dtype=bool_)
    for m in miss:
        idx.insert(m, (m if m<len(connxt) else 0) if injector else m-1)
        is_neigh[m] = 0
    #is_neigh[list(miss)] = 0
    if injector:
        # Injector
        diff = zeros(len(tub))
        for i in range(len(tub)):
            if is_neigh[i+1]:
                diff[idx[i+1]] = tub[i] - tub[idx[i+1]]
        diff[idx[0]] = wellrate - tub[idx[0]]
    else:
        # Producer
        diff = tub - is_neigh[1:]*tub[idx[1:]]
    return squeeze(diff)


def convert_surf(data):
    keys = ('CONOTUB', 'CONWTUB', 'CONGTUB')
    diff_rates = ((f'diff_{key}',rate_from_tubs(data[key], data['WELLPLT'][i], data['CONNXT'])) for i, key in enumerate(keys))
    #if data['CONWTUB'][0] > 0:
    if any(w<0 for w in data['WELLPLT'][:3]):
        typ = 'Injector'
    else:
        typ = 'Producer'
    # Extract RAT- and TUB-values from input data
    conrat = list((k,v) for k,v in data.items() if 'CON' in k and len(k)==7) # Do not include CON*TUBL values
    rates = flatten((a,b,c) for (a,b), c in zip(batched(conrat,2), diff_rates))
    header = (('date', data['DATE']), ('well', data['WELLETC'][1]+' '+data['WELLETC'][6]), ('type', typ)) #, ('CONNXT', data['CONNXT']))
    return dict( header + tuple(rates))

def convert_res(data):
    for d in data:
        keys = ('CONOTUBL', 'CONWTUBL', 'CONGTUBL')
        well_phase_rates = d['WELLPLT'][5]*(d['WELLPLT'][:3]/sum(d['WELLPLT'][:3]))
        diff_rates = ((f'diff_{key}',rate_from_tubs(d[key], well_phase_rates[i], d['CONNXT'])) for i, key in enumerate(keys))
        res_wellrate = d['WELLPLT'][5]
        if res_wellrate < 0:
            typ = 'Injector'
        else:
            typ = 'Producer'
        header = (('date', d['DATE']), ('well', d['WELLETC'][1]), ('type', typ), ('WELLPLT', d['WELLPLT']))
        yield dict( header + tuple(diff_rates))

def max_diff(data, well=None):
    for i, d in enumerate(data):
        if rates := convert_surf(d):
            if well and d['WELLETC'][1] != well:
                continue
            #pp.pprint(rates)
            diff = (abs(rates[f'{key}RAT'] - rates[f'diff_{key}TUB']) for key in ('CONO', 'CONW', 'CONG'))
            yield i, list(map(npmax, diff)), ('CONNXT', d['CONNXT']), ('WELLPLT', d['WELLPLT'][:]), rates
            # yield i, list(map(npmax, diff)), ('CONNXT', d['CONNXT']), ('WELLPLT', d['WELLPLT'][:]), ('I', d['CONIPOS']), ('J', d['CONJPOS']), ('K', d['CONKPOS']), rates
        else:
            print(f"{i}: Skipped well {d['WELLETC'][1]} at {d['DATE']}: ")
            print(f"{d['CONNXT']}\nCONORAT:{d['CONORAT']}\nCONOTUB:{d['CONOTUB']}")

def calc_res_from_surf_blocks(root):
    unrst = UNRST_file(root)
    oil = ('FLOOILI+', 'FLOOILJ+', 'FLOOILK+')
    wat = ('FLOWATI+', 'FLOWATJ+', 'FLOWATK+')
    gas = ('FLOGASI+', 'FLOGASJ+', 'FLOGASK+')
    conv = ('RS', 'RV', 'BO', 'BG', 'BW')
    for ihead, *filemap in unrst.blockdata('INTEHEAD', *oil, *wat, *gas, *conv):
        date = ihead[64:67]
        OI, OJ, OK,  WI, WJ, WK,  GI, GJ, GK,  RS, RV, BO, BG, BW = [array(d) for d in filemap]
        OIL = stack((OI, OJ, OK), axis=0)
        WAT = stack((WI, WJ, WK), axis=0)
        GAS = stack((GI, GJ, GK), axis=0)

        # From IORSim code
        # rate.oil   = Bo * (     STC_rate.oil - Rv*STC_rate.gas )/( 1.0 - Rs*Rv );
        # rate.gas   = Bg * ( -Rs*STC_rate.oil +    STC_rate.gas )/( 1.0 - Rs*Rv );
        # rate.water = Bw *       STC_rate.water;

        reso = BO * (     OIL - RV*GAS ) / ( 1.0 - RS*RV )
        resg = BG * ( -RS*OIL +    GAS ) / ( 1.0 - RS*RV )
        resw = BW *       WAT

        yield (date, reso, resw, resg)


#root = shared/'CoP_files/output/IORSIM_BASECASE_1971TO2022/IORSIM_BASECASE_1971TO2022'
root = shared/'CoP_files/input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'
#root = shared/'CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM'
rft = RFT_file(root)
keys = ('TIME', 'DATE', 'WELLETC', 'WELLPLT', 'CONNXT',
        'CONOTUB', 'CONOTUBL', 'CONORAT', 
        'CONWTUB', 'CONWTUBL', 'CONWRAT', 
        'CONGTUB', 'CONGTUBL', 'CONGRAT')
# Surface rates
rftdata = (dict(zip(keys, (array(d) for d in bd))) for bd in rft.blockdata(*keys))
#limit = 2e-0
#surf_data = next(islice(max_diff(rftdata), None, None))
# Check limit for all phases
#surf_data = next(islice((md for md in max_diff(rftdata) if any(d>limit for d in  md[1])), None, None))
# Check limit only for water
dmax = [-1]
def limit_phase(phase, limit):
    for md in max_diff(rftdata):
        diff = md[1][phase]
        dmax[0] = max(dmax[0], diff)
        if diff > limit:
            yield md
surf_data = next(islice(limit_phase(1, limit=0.5), 0, None), 'All is OK!')
#surf_data = next(islice(max_diff(rftdata, 'B06A'), 0, None))
#surf_data = next(islice(max_diff(rftdata), num, None))
#surf_data = next(islice((d for d in max_diff(rftdata) if d[-1]['type']=='Injector'), num, None))
pp.pprint(dmax)
pp.pprint(surf_data)


[5.8478908613324165e-06]
'All is OK!'


In [ ]:
from pathlib import Path
from numpy import array, stack
from IORlib.ECL import UNRST_file

home = Path.home()
shared = Path('/home/shared/owner')
root = shared/'CoP_files/input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'
unrst = UNRST_file(root)
oil = ('FLOOILI+', 'FLOOILJ+', 'FLOOILK+')
wat = ('FLOWATI+', 'FLOWATJ+', 'FLOWATK+')
gas = ('FLOGASI+', 'FLOGASJ+', 'FLOGASK+')
conv = ('RS', 'RV', 'BO', 'BG', 'BW')
for ihead, *filemap in unrst.blockdata('INTEHEAD', *oil, *wat, *gas, *conv):
    OI, OJ, OK,  WI, WJ, WK,  GI, GJ, GK,  RS, RV, BO, BG, BW = [array(d) for d in filemap]
    OIL = stack((OI, OJ, OK), axis=0)
    WAT = stack((WI, WJ, WK), axis=0)
    GAS = stack((GI, GJ, GK), axis=0)

    # From IORSim code
    # rate.oil   = Bo * (     STC_rate.oil - Rv*STC_rate.gas )/( 1.0 - Rs*Rv );
    # rate.gas   = Bg * ( -Rs*STC_rate.oil +    STC_rate.gas )/( 1.0 - Rs*Rv );
    # rate.water = Bw *       STC_rate.water;

    reso = BO * (     OIL - RV*GAS ) / ( 1.0 - RS*RV )
    resg = BG * ( -RS*OIL +    GAS ) / ( 1.0 - RS*RV )
    resw = BW *       WAT

    print(reso.shape)

    break

(3, 1284780)


In [ ]:
well = set()
for connxt, welletc in rft.blockdata('CONNXT', 'WELLETC'):
    if max(npdiff(array(connxt))) > 1:
        w = welletc[1]
        if w not in well:
            print(f'{w} {welletc[6]}: {connxt}')
        well.add(w)


In [ ]:
from datetime import datetime
from IORlib.ECL import UNRST_file

unrst = UNRST_file(root)
start = next(i for i,date in enumerate(unrst.dates()) if date >= datetime(1982, 10, 1))
cells = next(unrst.icellarray('FLRWATI+', 'FLRWATJ+', 'FLRWATK+', start=start))
[cells.flrwati[i,j,k]+cells.flrwatj[i,j,k]+cells.flrwatj[i,j,k] for i,j,k in zip(*failed[-1])]


In [ ]:
from datetime import datetime

smry = UNSMRY_file(root)
smrydata = smry.welldata(wells=('B21',), named=True)
list((d,r) for d,r in zip(smrydata.dates, smrydata.WWPR.B21) if d>=datetime(1982, 10, 1) )


In [ ]:
from IORlib.ECL import IX_input
inp = IX_input(root)
inp.findall('DATE "2-Jan')


In [ ]:
def well(rate, well='Inj_Comp'):
    return f'Well "{well}" {{\n    Constraints=[Constraint({rate} WATER_INJECTION_RATE)]\n}}'

def date(start, sec):
    return f'DATE "{start}:{sec:02d}"'
    
def soft_rate_inc(Q0, Q1, steps=10):
    dq = (Q1[1]-Q0[1])/steps
    rates = (Q0[1]+(i+1)*dq for i in range(steps))
    start = Q1[0]
    print(f'# Increase rate from {Q0[1]} to {Q1[1]} in {steps} steps of size {dq}')
    for i,rate in enumerate(rates):
        print(well(rate) + '\n' + date(start, i+1))

Q0 = ('1-Jan-2000 00:00' , 0.00090573269926)
Q1 = ('3-Jan-2000 21:42' , 0.0045286634963)
Q2 = ('3-Jan-2000 22:18' , 0.0090573269926)
Q3 = ('3-Jan-2000 22:36' , 0.0181146539852)
Q4 = ('3-Jan-2000 22:45' , Q0[1])
Q5 = ('11-Jan-2000 05:45', Q1[1])
Q6 = ('11-Jan-2000 08:45', Q2[1])
Q7 = ('11-Jan-2000 09:57', Q3[1])
Q8 = ('11-Jan-2000 10:33', Q0[1])

print('# JLV ----------------------------------------')
soft_rate_inc(Q3, Q4)
print('# JLV ----------------------------------------')


from ior2ecl import IORSim_output, IORSim_input
import matplotlib.pyplot as pl
%matplotlib widget
iorinp =IORSim_input(root)
#iorinp.wells()
out = IORSim_output(root)
ior_data = out.data('Prod_Com')
#ior_data.conc['CA']
pl.close('all')
fig, ax = pl.subplots(ncols=1, figsize=(10, 3.5))
#ax.semilogy(ior_data.days, ior_data.conc['CL'])
ax.plot(ior_data.days, ior_data.conc['CL'], '-o', mfc='None')


In [ ]:
data = smry.welldata(keys=('WWPR',), named=True)
#data.WWPR.Prod_Com
[f'{str(d).split(".")[0]}, {r:.8e}, {c}' for d,r,c in (zip(data.dates, data.WWPR.Prod_Com, ior_data.conc['CL']))]

In [ ]:
from IORlib.ECL import UNRST_file
unrst = UNRST_file(root)
next(unrst.section_blocks())
